# Lesson10: LangChainの主要モジュール1【Model I/O】：Output Parser

## 事前準備

OpenAI APIキーの設定

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

各種パッケージのインストール

In [ ]:
!pip install langchain==0.3.26 openai==1.91.0 langchain-community==0.3.26 langchain-openai==0.3.27 httpx==0.28.1 pydantic==2.11.7

## 記述用

In [ ]:
from langchain.output_parsers import StructuredOutputParser,ResponseSchema
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
schemas = [
    ResponseSchema(name="programming_language", description="プログラミング言語"),
    ResponseSchema(name="overview", description="プログラミング言語の説明")
]

In [ ]:
output_parser = StructuredOutputParser.from_response_schemas(schemas)

In [ ]:
format_instruction = output_parser.get_format_instructions()
print(format_instruction)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"programming_language": string  // プログラミング言語
	"overview": string  // プログラミング言語の説明
}
```


In [ ]:
prompt_template = PromptTemplate(
    template="友達に渡す誕生日プレゼントの案を5つ挙げてください。\n{format_instruction}",
    input_variables=[],
    partial_variables={"format_instruction": format_instruction}
)

prompt = prompt_template.format_prompt()
print(prompt.text)

友達に渡す誕生日プレゼントの案を5つ挙げてください。
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"programming_language": string  // プログラミング言語
	"overview": string  // プログラミング言語の説明
}
```


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=prompt.text),
]

output = llm(messages)
print(output.content)

/tmp/ipykernel_1373/3161566665.py:11: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  output = llm(messages)


```json
{
	"gift_ideas": [
		{
			"item": "カスタムマグカップ",
			"description": "友達の好きなデザインやメッセージを入れたオリジナルのマグカップ。毎日使ってもらえる実用的なプレゼント。"
		},
		{
			"item": "ハンドメイドのアクセサリー",
			"description": "手作りのネックレスやブレスレットなど、個性的で特別感のあるアクセサリー。友達のスタイルに合わせたものを選ぶと良い。"
		},
		{
			"item": "体験ギフト",
			"description": "料理教室やアートワークショップ、スパなどの体験を贈る。思い出に残る特別な時間を過ごしてもらえる。"
		},
		{
			"item": "本",
			"description": "友達の興味に合った本や、話題の新刊を選ぶ。読書が好きな友達にはぴったりのプレゼント。"
		},
		{
			"item": "パーソナライズドフォトアルバム",
			"description": "友達との思い出の写真を集めて作ったアルバム。特別な瞬間を振り返ることができる感動的なプレゼント。"
		}
	]
}
```


In [ ]:
from langchain.output_parsers.enum import EnumOutputParser
from enum import Enum
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
class Prefectures(Enum):
    PREFECTURES_1 = "北海道"
    PREFECTURES_2 = "京都府"
    PREFECTURES_3 = "沖縄県"

In [ ]:
output_parser = EnumOutputParser(enum=Prefectures)

In [ ]:
format_instruction = output_parser.get_format_instructions()
print(format_instruction)

Select one of the following options: 北海道, 京都府, 沖縄県


In [ ]:
prompt_template = PromptTemplate(
    template="観光におすすめの都道府県を1つ挙げてください。ただし、回答は都道府県名のみとしてください。\n{format_instruction}",
    input_variables=[],
    partial_variables={"format_instruction": format_instruction}
)

prompt = prompt_template.format_prompt()
print(prompt.text)

観光におすすめの都道府県を1つ挙げてください。ただし、回答は都道府県名のみとしてください。
Select one of the following options: 北海道, 京都府, 沖縄県


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=prompt.text),
]

output = llm(messages)
print(output.content)

京都府


In [ ]:
result = output_parser.parse(output.content)
print(result)

Prefectures.PREFECTURES_2


In [ ]:
from langchain.output_parsers import PydanticOutputParser

In [ ]:
from pydantic import BaseModel, Field

class TravelPlan(BaseModel):
    destinations: list[str] = Field(description="List of destinations included in the itinerary")
    activities: list[str] = Field(description="Planned activities at each destination")
    duration: int = Field(description="Total duration of the trip in days")
    budget: float = Field(description="Estimated budget for the trip in local currency")

In [ ]:
output_parser = PydanticOutputParser(pydantic_object=TravelPlan)

In [ ]:
format_instruction = output_parser.get_format_instructions()
print(format_instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"destinations": {"description": "List of destinations included in the itinerary", "items": {"type": "string"}, "title": "Destinations", "type": "array"}, "activities": {"description": "Planned activities at each destination", "items": {"type": "string"}, "title": "Activities", "type": "array"}, "duration": {"description": "Total duration of the trip in days", "title": "Duration", "type": "integer"}, "budget": {"description": "Estimated budget for the trip in local currency", "title": "Budget", "type": "number"}}, "required": ["

In [ ]:
from langchain.prompts import PromptTemplate

template = """
次のフォーマットで旅行プランを作成してください。

{format_instruction}

旅行のテーマ: {theme}
"""

prompt_template = PromptTemplate(
    template=template,
    input_variables=["theme"],
    partial_variables={"format_instruction": format_instruction}
)

In [ ]:
prompt = prompt_template.format(theme="家族で楽しむリゾート旅行")
print(prompt)


次のフォーマットで旅行プランを作成してください。

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"destinations": {"description": "List of destinations included in the itinerary", "items": {"type": "string"}, "title": "Destinations", "type": "array"}, "activities": {"description": "Planned activities at each destination", "items": {"type": "string"}, "title": "Activities", "type": "array"}, "duration": {"description": "Total duration of the trip in days", "title": "Duration", "type": "integer"}, "budget": {"description": "Estimated budget for the trip in local currency", "title": "Budget", "type":

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=prompt),
]

output = llm(messages)
print(output.content)

```json
{
  "destinations": [
    "ハワイ",
    "バリ島",
    "グアム"
  ],
  "activities": [
    "ビーチでのんびりする",
    "シュノーケリング",
    "観光ツアー",
    "地元の料理を楽しむ",
    "スパ体験",
    "ハイキング"
  ],
  "duration": 7,
  "budget": 2000000
}
```


In [ ]:
result = output_parser.parse(output.content)
print(result)

destinations=['ハワイ', 'バリ島', 'グアム'] activities=['ビーチでのんびりする', 'シュノーケリング', '観光ツアー', '地元の料理を楽しむ', 'スパ体験', 'ハイキング'] duration=7 budget=2000000.0
